# Imports

In [1]:
import json
import numpy as np
import random
from google.colab import files, drive
import os
from datasets import Dataset
from collections import Counter
from transformers import AutoTokenizer, AutoModelForTokenClassification, TrainingArguments, Trainer, AutoModelForSeq2SeqLM
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score, classification_report, precision_recall_fscore_support
import torch

In [2]:
SEED=42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [3]:
drive.mount('/content/drive')

Mounted at /content/drive


# Data Loading





In [4]:
uploaded=files.upload()

Saving data.json to data.json
Saving test_data.json to test_data.json


In [ ]:
print('Paths:', os.listdir())

Paths: ['.config', 'data.json', 'test_data.json', 'drive', 'sample_data']


In [5]:
with open("data.json", "r") as f:
    data=json.load(f)
with open("test_data.json", "r") as f:
    test_data=json.load(f)

# Data Understanding

In [6]:
dataset=Dataset.from_list(data)

In [ ]:
print('First entry of datset:', dataset[0])
print('Dataset Features:',dataset.features)
print('Dataset Length:', len(dataset))

First entry of datset: {'lang': 'en', 'ner_tags': ['O', 'O', 'O', 'O', 'B-PER', 'I-PER', 'O', 'O', 'O', 'O', 'B-PER', 'I-PER', 'O', 'O', 'O', 'O', 'O', 'B-PER', 'I-PER', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O'], 'sequence': 'Since then , only Terry Bradshaw in 147 games , Joe Montana in 139 games , and Tom Brady in 131 games have reached 100 wins more quickly .', 'tokens': ['Since', 'then', ',', 'only', 'Terry', 'Bradshaw', 'in', '147', 'games', ',', 'Joe', 'Montana', 'in', '139', 'games', ',', 'and', 'Tom', 'Brady', 'in', '131', 'games', 'have', 'reached', '100', 'wins', 'more', 'quickly', '.']}
Dataset Features: {'lang': Value('string'), 'ner_tags': List(Value('string')), 'sequence': Value('string'), 'tokens': List(Value('string'))}
Dataset Length: 28516


First entry of Test dataset: {'lang': 'en', 'ner_tags': ['O', 'O', 'O', 'O', 'B-PER', 'I-PER', 'O', 'O', 'O', 'O', 'B-PER', 'I-PER', 'O', 'O', 'O', 'O', 'O', 'B-PER', 'I-PER', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', '

In [ ]:
name_counts=0      #counting number of sample containing name
for sample in data:
    if any("PER" in tag for tag in sample["ner_tags"]):
        name_counts+=1

percentage = name_counts / len(data) * 100
print('Percentage of sample containing name:',percentage)

Percentage of sample containing name: 100.0


In [ ]:
label_counts=Counter()        #checking class distribution
for sample in data:
    label_counts.update(sample["ner_tags"])

print('Label counts:', label_counts)

Label counts: Counter({'O': 639055, 'B-PER': 40264, 'I-PER': 29466})


# Encoder based transformer Model (Bert)

Data Preprocessing

In [7]:
def convert_labels(sample):         #converting custom labels to model specific labels
    new_tags=[]
    for tag in sample["ner_tags"]:
        if tag=="B-PER":
            new_tags.append("B-NAME")
        elif tag=="I-PER":
            new_tags.append("I-NAME")
        else:
            new_tags.append(tag)

    sample["ner_tags"]=new_tags
    return sample

data=[convert_labels(sample) for sample in data]
test_data=[convert_labels(sample) for sample in test_data]

In [ ]:
def generate_email(name):       #generating synthetic emails
    domains=["gmail.com", "yahoo.com", "outlook.com"]
    return name.lower() + str(random.randint(10,999)) + "@" + random.choice(domains)

In [ ]:
def add_email(sample):
    if random.random() < 0.25:     #injecting emails in 25% data
        name_tokens = []
        for token, tag in zip(sample["tokens"], sample["ner_tags"]):
            if "NAME" in tag:        # find a NAME token to base email on
                name_tokens.append(token)

        if len(name_tokens) > 0:
            name=name_tokens[0]
            email=generate_email(name)

            sample["tokens"].insert(-1, email)
            sample["ner_tags"].insert(-1, "B-EMAIL")

    return sample

data=[add_email(sample) for sample in data]

In [ ]:
for i in range(2):
    print('Tokens:', data[i]["tokens"])
    print('NER Tags:', data[i]["ner_tags"])
    print("------")

['Since', 'then', ',', 'only', 'Terry', 'Bradshaw', 'in', '147', 'games', ',', 'Joe', 'Montana', 'in', '139', 'games', ',', 'and', 'Tom', 'Brady', 'in', '131', 'games', 'have', 'reached', '100', 'wins', 'more', 'quickly', '.']
['O', 'O', 'O', 'O', 'B-NAME', 'I-NAME', 'O', 'O', 'O', 'O', 'B-NAME', 'I-NAME', 'O', 'O', 'O', 'O', 'O', 'B-NAME', 'I-NAME', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']
------
['He', 'was', 'portrayed', 'by', 'Anthony', 'Perkins', 'in', 'the', '1960', 'version', 'of', '"', 'Psycho', '"', 'directed', 'by', 'Alfred', 'Hitchcock', 'and', 'the', '"', 'Psycho', '"', 'franchise', 'anthony291@gmail.com', '.']
['O', 'O', 'O', 'O', 'B-NAME', 'I-NAME', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-NAME', 'I-NAME', 'O', 'O', 'O', 'O', 'O', 'O', 'B-EMAIL', 'O']
------
['The', 'egg', 'eventually', 'hatches', ',', 'revealing', 'a', 'baby', 'Sharptooth', 'sharptooth764@gmail.com', '.']
['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-NAME', 'B-EMAIL', 'O']
------
['In'

In [ ]:
label_counts=Counter()        #checking class distribution after generating emails
for sample in data:
    label_counts.update(sample["ner_tags"])

print('Label Counts:', label_counts)

Counter({'O': 639055, 'B-NAME': 40264, 'I-NAME': 29466, 'B-EMAIL': 7144})


Saving and Reloading processed dataset



In [ ]:
with open("/content/drive/MyDrive/processed_data.json", "w") as f:    #Saving to Drive
    json.dump(data, f)

In [8]:
with open("/content/drive/MyDrive/processed_data.json", "r") as f:     #Loading from Drive
    data = json.load(f)

Train-Test Split

In [9]:
data=list(data)           # 80-20 train-validation split
random.shuffle(data)
split_index=int(0.8 * len(data))
train_data=data[:split_index]
val_data=data[split_index:]
print("Train %:", len(train_data) / len(data) * 100)
print("Validation %:", len(val_data) / len(data) * 100)

Train %: 79.99719455744143
Validation %: 20.002805442558564


Tokenization

In [10]:
tokenizer=AutoTokenizer.from_pretrained("bert-base-cased")        #using bert-tokenizer to tokenize given data
def tokenize(example):
    return tokenizer(
        example["tokens"],
        is_split_into_words=True,
        truncation=True,
        padding="max_length",
        max_length=128
    )
train_tokens=[tokenize(x) for x in train_data]
val_tokens=[tokenize(x) for x in val_data]
test_tokens=[tokenize(x) for x in test_data]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
for i in range(2):
    tokens = tokenizer.convert_ids_to_tokens(train_tokens[i]["input_ids"])
    print('Tokens:', tokens)
    print("-----")

Tokens: ['[CLS]', 'On', 'the', 'other', 'hand', ',', 'So', '##crates', 'believed', 'that', 'democracy', 'without', 'educated', 'masses', '(', 'educated', 'in', 'the', 'more', 'broader', 'sense', 'of', 'being', 'knowledge', '##able', 'and', 'responsible', ')', 'would', 'only', 'lead', 'to', 'pop', '##uli', '##sm', 'being', 'the', 'criteria', 'to', 'become', 'an', 'elected', 'leader', 'and', 'not', 'competence', '.', '[SEP]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD

Label Alignment

In [11]:
labels = ["O", "B-NAME", "I-NAME", "B-EMAIL"]
label2id = {label: i for i, label in enumerate(labels)}
id2label = {i: label for label, i in label2id.items()}

In [12]:
def align_labels(tokenized_inputs, original_data):       #ensures subword get one label

    aligned_labels=[]

    for i, sample in enumerate(original_data):

        word_ids=tokenized_inputs[i].word_ids()
        labels=[]

        previous_word_id=None

        for word_id in word_ids:

            if word_id is None:
                labels.append(-100)  # special tokens

            elif word_id!=previous_word_id:    # first token of word
                labels.append(label2id[sample["ner_tags"][word_id]])

            else:
                labels.append(-100)  # subword token

            previous_word_id=word_id

        aligned_labels.append(labels)

    return aligned_labels



train_labels=align_labels(train_tokens, train_data)
val_labels=align_labels(val_tokens, val_data)
test_labels=align_labels(test_tokens, test_data)

In [13]:
def pad_labels(labels, max_length=128):     #setting max of labels list
    return labels[:max_length] + [-100] * (max_length - len(labels))

train_labels=[pad_labels(x) for x in train_labels]
val_labels=[pad_labels(x) for x in val_labels]
test_labels=[pad_labels(x) for x in test_labels]

Model Setup

In [ ]:
bert_model=AutoModelForTokenClassification.from_pretrained(   #Loading fresh bert model
    "bert-base-cased",
    num_labels=len(labels),
    label2id=label2id,
    id2label=id2label
)
bert_model.config.label2id=label2id
bert_model.config.id2label=id2label

print(bert_model.config)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized beca

BertConfig {
  "add_cross_attention": false,
  "architectures": [
    "BertForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": null,
  "classifier_dropout": null,
  "dtype": "float32",
  "eos_token_id": null,
  "gradient_checkpointing": false,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "id2label": {
    "0": "O",
    "1": "B-NAME",
    "2": "I-NAME",
    "3": "B-EMAIL"
  },
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "is_decoder": false,
  "label2id": {
    "B-EMAIL": 3,
    "B-NAME": 1,
    "I-NAME": 2,
    "O": 0
  },
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token_id": 0,
  "position_embedding_type": "absolute",
  "tie_word_embeddings": true,
  "transformers_version": "5.0.0",
  "type_vocab_size": 2,
  "use_cache": true,
  "vocab_size": 28996
}



Training

In [14]:
train_dataset=Dataset.from_dict({
    "input_ids": [x["input_ids"] for x in train_tokens],
    "attention_mask": [x["attention_mask"] for x in train_tokens],
    "labels": train_labels
})

val_dataset=Dataset.from_dict({
    "input_ids": [x["input_ids"] for x in val_tokens],
    "attention_mask": [x["attention_mask"] for x in val_tokens],
    "labels": val_labels
})
test_dataset=Dataset.from_dict({
    "input_ids": [x["input_ids"] for x in test_tokens],
    "attention_mask": [x["attention_mask"] for x in test_tokens],
    "labels": test_labels})

In [19]:
training_args=TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    eval_strategy="epoch",
    logging_dir="./logs",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [20]:
trainer=Trainer(
    model=bert_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,0.011111,0.007023
2,0.003792,0.007190


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=2852, training_loss=0.01263816953541353, metrics={'train_runtime': 1170.353, 'train_samples_per_second': 38.983, 'train_steps_per_second': 2.437, 'total_flos': 2980404697669632.0, 'train_loss': 0.01263816953541353, 'epoch': 2.0})

Saving and Reloading trained model

In [ ]:
trainer.save_model("/content/drive/MyDrive/pii_model")      #saving trained model
tokenizer.save_pretrained("/content/drive/MyDrive/pii_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/drive/MyDrive/pii_model/tokenizer_config.json',
 '/content/drive/MyDrive/pii_model/tokenizer.json')

In [15]:
bert_model = AutoModelForTokenClassification.from_pretrained("/content/drive/MyDrive/pii_model")   #loading trained model
tokenizer = AutoTokenizer.from_pretrained("/content/drive/MyDrive/pii_model")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Predictions




In [16]:
extra_test=['John Doe lives in New York. You can contact him at john.doe@gmail.com','Sarah Ahmed works at a software company. Her email is sarah.ahmed@yahoo.com','Michael Smith recently joined the team. Reach him at m.smith@outlook.com','Ayesha Khan is a university student. Email: ayesha.khan123@gmail.com','David Johnson will attend the meeting. His email is david.johnson@company.com','Ali Raza is our project manager. Contact him at ali.raza@business.org','Emily Brown submitted her report. You can email her at emily.brown@mail.com','Usman Tariq is responsible for this task. His email is usman.tariq@gmail.com','Jessica Lee updated the document. Reach her at jessica.lee@workplace.net','Ahmed Hassan is available for support. Email him at ahmed.hassan@helpdesk.com']
# testing on independent dataset containing both names and emails
def predict_extra_test(text):
    inputs=tokenizer(text, return_tensors="pt")

    with torch.no_grad():
        outputs=bert_model(**inputs)

    predictions=outputs.logits.argmax(dim=2)

    pred_labels=[id2label[p.item()] for p in predictions[0]]
    tokens=tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

    return tokens, pred_labels


Masking

In [17]:
def clean_and_mask(tokens, labels):    #masking output based on predicted labels
    words=[]
    current_word=""
    current_label=None

    for token, label in zip(tokens, labels):
        if token in ["[CLS]", "[SEP]", "[PAD]"]:    # Skip special tokens
            continue

        if token.startswith("##"):        # Merge subwords
            current_word+=token[2:]
        else:
            if current_word:
                words.append((current_word, current_label))
            current_word=token
            current_label=label

    if current_word:
        words.append((current_word, current_label))

    final_text=[]    # Apply masking
    i=0

    while i < len(words):
        word, label=words[i]

        if label=="B-NAME":
            final_text.append("[NAME]")
            i+=1
            while i < len(words) and words[i][1]=="I-NAME":
                i+=1

        elif label=="B-EMAIL":
            final_text.append("[EMAIL]")
            i+=1
            while i < len(words) and (
                "@" in words[i][0] or
                "." in words[i][0] or
                words[i][0].isalnum()
            ):
                i+=1

        else:
            final_text.append(word)
            i+=1

    return " ".join(final_text)

In [ ]:
for text in extra_test:       #Testing on data with emails
    tokens,pred_labels=predict_extra_test(text)

    print("Original:", text)
    print("Masked  :", clean_and_mask(tokens,pred_labels))
    print("\n")

Original: John Doe lives in New York. You can contact him at john.doe@gmail.com
Masked  : [NAME] lives in New York . You can contact him at [EMAIL]


Original: Sarah Ahmed works at a software company. Her email is sarah.ahmed@yahoo.com
Masked  : [NAME] works at a software company . Her email is [EMAIL]


Original: Michael Smith recently joined the team. Reach him at m.smith@outlook.com
Masked  : [NAME] recently joined the team . Reach him at [EMAIL]


Original: Ayesha Khan is a university student. Email: ayesha.khan123@gmail.com
Masked  : [NAME] is a university student . Email : [EMAIL]


Original: David Johnson will attend the meeting. His email is david.johnson@company.com
Masked  : [NAME] will attend the meeting . His email is [EMAIL]


Original: Ali Raza is our project manager. Contact him at ali.raza@business.org
Masked  : [NAME] is our project manager . Contact him at ali . [EMAIL]


Original: Emily Brown submitted her report. You can email her at emily.brown@mail.com
Masked  : [

Evaluation

In [21]:
predictions, labels, _=trainer.predict(test_dataset)
preds=np.argmax(predictions, axis=2)

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


In [22]:
true_labels=[]
true_preds=[]
for pred_seq, label_seq in zip(preds, labels):
    for p, l in zip(pred_seq, label_seq):
        if l!=-100:
            true_labels.append(l)
            true_preds.append(p)

id2label=bert_model.config.id2label

true_labels_named=[id2label[i] for i in true_labels]
true_preds_named=[id2label[i] for i in true_preds]

bert_report=classification_report(
    true_labels,
    true_preds,
    output_dict=True
)

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [23]:
precision=precision_score(true_labels, true_preds, average="weighted")      #evaluation performance
recall=recall_score(true_labels, true_preds, average="weighted")
f1=f1_score(true_labels, true_preds, average="weighted")
accuracy=accuracy_score(true_labels, true_preds)

print("Precision:", precision)
print("Recall:", recall)
print("F1-score:", f1)
print("Accuracy:", accuracy)

Precision: 0.9973661189963662
Recall: 0.9973452547865861
F1-score: 0.9973547558818553
Accuracy: 0.9973452547865861


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
